# 01 — Molecular Lab

**No API key required.** Encode molecules, compute descriptors, generate 3D conformers, and compare structures.

Capabilities demonstrated:
- Morgan ECFP4 fingerprint encoding (2048-bit)
- Physicochemical descriptors (MW, LogP, TPSA, QED, SA score, Lipinski)
- 3D conformer generation (ETKDG + UFF energy minimisation)
- Tanimoto similarity comparison
- Batch encoding

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from chemvision.models.mol_encoder import MolecularEncoder

enc = MolecularEncoder()
print('MolecularEncoder ready')

## 1.1 — Encode a molecule to a fingerprint

In [ ]:
ASPIRIN = 'CC(=O)Oc1ccccc1C(=O)O'

emb = enc.encode(ASPIRIN)
print(f'Shape: {emb.shape}')
print(f'Non-zero bits: {int(emb.sum())} / {len(emb)} ({emb.sum()/len(emb)*100:.1f}% density)')

# Visualise fingerprint as a heatmap
fig, ax = plt.subplots(figsize=(14, 2))
ax.imshow(emb.reshape(32, 64), aspect='auto', cmap='Blues', interpolation='nearest')
ax.set_title(f'Morgan ECFP4 — {ASPIRIN}')
ax.set_xlabel('Bit index (mod 64)')
ax.set_ylabel('Row')
plt.tight_layout()
plt.show()

## 1.2 — Compute physicochemical descriptors

In [ ]:
molecules = {
    'Aspirin': 'CC(=O)Oc1ccccc1C(=O)O',
    'Ibuprofen': 'CC(C)Cc1ccc(cc1)C(C)C(=O)O',
    'Caffeine': 'Cn1cnc2c1c(=O)n(c(=O)n2C)C',
    'Paracetamol': 'CC(=O)Nc1ccc(O)cc1',
    'Metformin': 'CN(C)C(=N)NC(=N)N',
    'Penicillin V': 'CC1(C(N2C(S1)C(C2=O)NC(=O)COc3ccccc3)C(=O)O)C',
}

import pandas as pd

rows = []
for name, smi in molecules.items():
    d = enc.compute_descriptors(smi)
    rows.append({
        'Name': name, 'SMILES': smi,
        'MW': f'{d.mw:.1f}', 'LogP': f'{d.logp:.2f}',
        'TPSA': f'{d.tpsa:.1f}', 'QED': f'{d.qed:.3f}',
        'HBD': d.hbd, 'HBA': d.hba,
        'Rings': d.rings, 'Lipinski': 'PASS' if d.lipinski_pass else 'FAIL',
    })

pd.DataFrame(rows)

## 1.3 — Generate a 3D conformer

In [ ]:
conf = enc.generate_conformer(ASPIRIN)
print(f'Success: {conf.success}')
print(f'Atoms: {conf.num_atoms}')
print(f'UFF energy: {conf.energy_kcal:.1f} kcal/mol')

coords = np.array(conf.coordinates)

# 2D scatter (x vs y)
elem_map = {1: 'H', 6: 'C', 7: 'N', 8: 'O'}
colors = {1: 'lightgray', 6: 'black', 7: 'blue', 8: 'red'}

fig, ax = plt.subplots(figsize=(8, 6))
for z_val in set(conf.atomic_numbers):
    mask = [z == z_val for z in conf.atomic_numbers]
    pts = coords[mask]
    ax.scatter(pts[:, 0], pts[:, 1], c=colors.get(z_val, 'gray'),
              s=100 if z_val > 1 else 30, label=elem_map.get(z_val, str(z_val)),
              edgecolors='darkgray', zorder=3)
ax.set_xlabel('x (Angstrom)')
ax.set_ylabel('y (Angstrom)')
ax.set_title(f'Aspirin 3D conformer (2D projection)')
ax.legend()
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 1.4 — Tanimoto similarity matrix

In [ ]:
names = list(molecules.keys())
smiles_list = list(molecules.values())
n = len(names)

sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = enc.tanimoto(smiles_list[i], smiles_list[j])

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(sim_matrix, cmap='YlOrRd', vmin=0, vmax=1)
ax.set_xticks(range(n)); ax.set_xticklabels(names, rotation=45, ha='right')
ax.set_yticks(range(n)); ax.set_yticklabels(names)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=9)
fig.colorbar(im, label='Tanimoto similarity')
ax.set_title('Pairwise Tanimoto similarity')
plt.tight_layout()
plt.show()

## 1.5 — Batch encoding

In [ ]:
matrix = enc.encode_batch(smiles_list)
print(f'Batch shape: {matrix.shape}')  # (6, 2048)
print(f'Mean bits per molecule: {matrix.sum(axis=1).mean():.0f}')